<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Computer-Networks/09-network-measurement-operations-troubleshooting.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Computer Networks guideline](Computer-Networks.html)


## **Network Measurement, Operations, and Troubleshooting**

A protocol description explains what a correct implementation should do. Operations begins when a real user says, "the site is slow," while links are redundant, routes are asymmetric, packets are encrypted, and every dashboard summarizes a different part of the system. The operational task is not to run as many tools as possible. It is to turn an imprecise symptom into a **falsifiable explanation**, collect evidence from useful vantage points, make the smallest reversible change, and verify that the original user-facing behavior recovered.

Three distinctions organize this chapter:

- **Measurement** assigns a quantity, unit, population, time window, and uncertainty to an observation.
- **Monitoring** repeats selected measurements so that change can be detected over time.
- **Troubleshooting** uses observations to discriminate among competing causes and decide what to test or change next.

A green dashboard is not proof that every user is healthy, and a red counter is not automatically the cause of an incident. Counters, active probes, packet captures, flow records, logs, and traces are partial views produced at particular points in the system. Reliable diagnosis comes from understanding what each view can observe, what it cannot observe, and which independent evidence should agree if a hypothesis is true.

::: {.callout-warning}
Active probes, captures, throughput tests, and failure injection must be limited to systems for which explicit authorization exists. A controlled throughput test can consume real capacity; a capture can contain credentials or personal data; and an apparently harmless scan can trigger defenses or violate policy. The examples below use synthetic data or local in-memory packets.
:::

### **From Operational Question to Measurement Plan**

#### **Reachability, Performance, Reliability, and Security Questions**

The word "network problem" hides several different questions. **Reachability** asks whether a packet exchange can complete between named endpoints under a particular protocol. **Performance** asks how much time or capacity an operation consumes. **Reliability** asks how consistently the operation succeeds across users and time. A **security** question asks whether an observed path, peer, configuration, or event satisfies an intended policy. These questions need different evidence.

For example, an ICMP echo reply supports the narrow claim that one ICMP request and reply traversed the observed paths. It does not prove that TCP port 443 is reachable, that the certificate is valid, that the HTTP application is healthy, or that another region follows the same path. Start with the user-visible operation and success condition, then descend to narrower protocol questions.

#### **Metrics, Units, Baselines, and Success Criteria**

A metric is incomplete without a **unit** and an **aggregation rule**. "Latency is 80" is meaningless until it becomes "the p95 client-observed checkout duration was 80 ms over five-minute windows." A current value also needs a baseline: the same population before a change, an unaffected control region, a seasonal forecast, or a documented objective. A good success criterion is decided before changing the system, such as "p95 TLS setup returns within 10% of baseline for two consecutive windows while error rate does not increase."

Means can conceal a harmed minority. Operational latency is usually reported as a distribution, with median for the typical case and p95 or p99 for the tail. Counts should include their denominator. A loss count of 100 packets is severe at 1,000 sent and negligible at 100 million sent.

#### **Measurement Point, Time Window, and Independent Unit**

The **vantage point** determines which portion of a path is visible. A server-side timer excludes the client access network and may begin after a reverse proxy has already queued the request. A host packet capture may see data before NIC offload, whereas a network TAP sees wire packets. A flow exporter after NAT records different endpoint tuples from a client capture before NAT.

The time window controls sensitivity. A one-minute average reacts quickly but is noisy; a one-hour average can hide a ten-minute outage. The independent unit also matters. Ten thousand requests from one persistent connection are not ten thousand independent path samples. Useful units may be users, connections, sessions, devices, sites, or repeated experimental runs.

#### **Active vs Passive Measurement**

**Active measurement** injects controlled traffic, such as ICMP probes, synthetic HTTPS transactions, or an iperf3 stream. It provides known inputs and can test a path even when no user traffic exists, but the probe may use a different protocol, priority, endpoint, or route from the application and can perturb the network.

**Passive measurement** observes existing activity through counters, flow records, logs, traces, or captures. It reflects real workloads but inherits their changing mixture and may not contain the variables needed to isolate cause. Mature investigations combine both: passive telemetry finds who is affected and when; a carefully matched active test varies one factor; packet or trace evidence explains the mechanism.

![A measurement plan turns a symptom into a falsifiable hypothesis, metric, vantage point, sampling design, and decision rule.](assets/measurement-plan-funnel.svg){fig-alt="Six stages from operational symptom to hypothesis metric vantage point sampling and decision" width="96%"}

In [1]:
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class MeasurementPlan:
    question: str
    hypothesis: str
    predicted_evidence: str
    falsifier: str
    metric: str
    unit: str
    population: str
    vantage_point: str
    window_minutes: int
    independent_unit: str
    baseline: str
    success_rule: str
    active_rate_per_second: Optional[float] = None

    def validate(self):
        """Return planning omissions before any traffic is generated."""
        problems = []
        required_text = {
            "hypothesis": self.hypothesis,
            "predicted evidence": self.predicted_evidence,
            "falsifier": self.falsifier,
            "metric": self.metric,
            "unit": self.unit,
            "population": self.population,
            "vantage point": self.vantage_point,
            "independent unit": self.independent_unit,
            "baseline": self.baseline,
            "success rule": self.success_rule,
        }
        for label, value in required_text.items():
            if not value.strip():
                problems.append(f"missing {label}")
        if self.window_minutes <= 0:
            problems.append("window must be positive")
        if self.active_rate_per_second is not None and self.active_rate_per_second <= 0:
            problems.append("active probe rate must be positive")
        return problems


plan = MeasurementPlan(
    question="Why did checkout become slower for Sydney mobile users?",
    hypothesis="a routing change added an inter-region path before the TLS endpoint",
    predicted_evidence="TLS and TCP phases rise together; server processing is unchanged",
    falsifier="handshake RTT is unchanged while server span duration rises",
    metric="client p95 TCP+TLS setup duration",
    unit="milliseconds",
    population="Sydney mobile checkout sessions",
    vantage_point="synthetic clients in two Sydney access networks",
    window_minutes=10,
    independent_unit="new transport connection",
    baseline="same weekday/hour plus Melbourne control probes",
    success_rule="p95 returns within 10% of baseline for two windows",
    active_rate_per_second=0.05,  # one probe every 20 seconds per vantage point
)

print("Plan valid:", not plan.validate())
print("Predicted evidence:", plan.predicted_evidence)
print("Would reject hypothesis if:", plan.falsifier)

Plan valid: True
Predicted evidence: TLS and TCP phases rise together; server processing is unchanged
Would reject hypothesis if: handshake RTT is unchanged while server span duration rises


The plan does not guarantee that the hypothesis is correct. It prevents the investigation from quietly changing its metric, population, or success threshold after seeing the data. The explicit falsifier is especially important: evidence collection should be capable of proving the current explanation wrong.

### **Core Network Metrics**

#### **Latency, RTT, and One-Way Delay**

**Latency** is elapsed time between two defined events. A packet's one-hop delay can be decomposed as

$$
d_{hop}=d_{proc}+d_{queue}+d_{trans}+d_{prop},
$$

where processing checks and forwards the packet, queueing waits for service, transmission places $L$ bits onto a link at rate $R$ so that $d_{trans}=L/R$, and propagation carries the signal across distance $D$ at speed $v$ so that $d_{prop}=D/v$. Only queueing varies rapidly with competing traffic; buying a faster access link does not remove geographic propagation delay.

**Round-trip time (RTT)** measures from an event at one endpoint to a related response at the same endpoint, so one clock is sufficient. It includes forward path, remote processing, and reverse path, which may differ. Dividing RTT by two is not a reliable one-way delay estimate unless both directions are known to be symmetric. **One-way delay** requires synchronized clocks and a precise definition of send and receive timestamps; clock offset can otherwise be mistaken for network delay.

Application latency is another decomposition. An HTTPS duration may include DNS resolution, connection setup, TLS, proxy queueing, server work, and body transfer. Preserve phase-level timings because identical totals can imply different remedies.

![Packet and application delay are sums of different mechanisms.](assets/network-latency-decomposition.svg){fig-alt="Packet path delay components and an HTTPS request timing waterfall" width="96%"}

#### **Loss, Duplication, Reordering, and Corruption**

For a defined packet stream with unique sequence identifiers,

$$
\text{loss rate}=\frac{N_{sent}-N_{unique\ received}}{N_{sent}}.
$$

The definition requires a pairing rule and observation interval. Packets arriving after the interval may be classified as lost even though they are merely late. A duplicate repeats an already observed identifier. Reordering occurs when arrival order differs from send order, but the chosen metric matters: displaced packet count, reordering depth, and reordering delay answer different questions. Link-layer corruption is often detected and discarded below an IP capture, so an absence of corrupt packets in Wireshark does not prove that the physical link had no errors; interface CRC counters may contain that evidence.

TCP retransmissions are not identical to network loss. A sender can retransmit after a delayed ACK, packet reordering, or a timeout caused by queueing. Conversely, link-layer recovery can hide radio loss from TCP while increasing latency.

#### **Throughput, Goodput, Utilization, and Capacity**

**Throughput** is the rate of transferred bits at a defined layer. **Goodput** counts useful application payload delivered once, excluding headers, retransmissions, duplicates, and application framing overhead:

$$
\text{goodput}=\frac{\text{unique useful payload bits}}{\text{measurement duration}}.
$$

**Utilization** divides observed rate by a stated link capacity. A 1 Gbit/s interface carrying 700 Mbit/s has 70% average utilization over that interval, but microbursts can still overflow a shallow queue. **Available capacity** is the additional rate a path can sustain under stated conditions; it is not the same as the narrowest link's configured rate. A single TCP flow may be limited by RTT, loss, receive window, host CPU, or congestion-control state before reaching path capacity.

#### **Jitter, Tail Latency, and Availability**

In IP performance measurement, delay variation is usually the difference between selected one-way delays. Real-time applications often care about inter-arrival variation because a playout buffer must absorb it. Avoid reporting "jitter" without naming the algorithm and units; RTP's smoothed inter-arrival jitter, standard deviation, and percentile spread are different quantities.

Tail latency reports high percentiles such as p95 or p99. A percentile is a property of a population and window, not a property of one request. **Availability** should be tied to a valid user operation:

$$
\text{availability}=\frac{\text{good eligible events}}{\text{all eligible events}}.
$$

Define exclusions before observing an incident. Removing failed requests because they timed out before reaching the server would produce an optimistic SLI.

In [2]:
from collections import Counter
from statistics import mean, median


def percentile(values, q):
    """Linearly interpolate a percentile for a small teaching dataset."""
    ordered = sorted(values)
    position = (len(ordered) - 1) * q
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    fraction = position - lower
    return ordered[lower] * (1 - fraction) + ordered[upper] * fraction


# One row per sent probe. None means no response inside the observation window.
probes = [
    {"seq": 1, "rtt_ms": 21.0, "arrival_order": 1, "wire_bytes": 1240, "payload_bytes": 1100},
    {"seq": 2, "rtt_ms": 23.5, "arrival_order": 2, "wire_bytes": 1240, "payload_bytes": 1100},
    {"seq": 3, "rtt_ms": None, "arrival_order": None, "wire_bytes": 1240, "payload_bytes": 0},
    {"seq": 4, "rtt_ms": 85.0, "arrival_order": 4, "wire_bytes": 2480, "payload_bytes": 1100},  # retransmitted once
    {"seq": 5, "rtt_ms": 27.0, "arrival_order": 3, "wire_bytes": 1240, "payload_bytes": 1100},
    {"seq": 6, "rtt_ms": 25.0, "arrival_order": 5, "wire_bytes": 1240, "payload_bytes": 1100},
]

received = [probe for probe in probes if probe["rtt_ms"] is not None]
rtts = [probe["rtt_ms"] for probe in received]
loss_rate = 1 - len(received) / len(probes)
reordered = sum(
    current["seq"] < previous["seq"]
    for previous, current in zip(
        sorted(received, key=lambda p: p["arrival_order"]),
        sorted(received, key=lambda p: p["arrival_order"])[1:],
    )
)

duration_seconds = 0.50
throughput_mbps = sum(p["wire_bytes"] for p in probes) * 8 / duration_seconds / 1e6
goodput_mbps = sum(p["payload_bytes"] for p in probes) * 8 / duration_seconds / 1e6

print(f"received/lost: {len(received)}/{len(probes) - len(received)}")
print(f"loss rate: {loss_rate:.1%}; adjacent arrival inversions: {reordered}")
print(f"RTT median/p95: {median(rtts):.1f}/{percentile(rtts, 0.95):.1f} ms")
print(f"throughput/goodput: {throughput_mbps:.3f}/{goodput_mbps:.3f} Mbit/s")
print("overhead and retransmission fraction:", f"{1 - goodput_mbps / throughput_mbps:.1%}")

received/lost: 5/1
loss rate: 16.7%; adjacent arrival inversions: 1
RTT median/p95: 25.0/73.4 ms
throughput/goodput: 0.139/0.088 Mbit/s
overhead and retransmission fraction: 36.6%


The small sample deliberately contains loss, reordering, and a retransmission. The p95 is far above the median even though most probes are fast. In a production report, state the sample count and confidence: a p99 computed from twenty requests is not a stable tail estimate.

### **Packet Capture and Protocol Analysis**

#### **Capture Points and Interface Selection**

A packet capture is a timestamped observation at one interface, namespace, or mirror. Before capturing, draw the path and mark where encryption, NAT, load balancing, tunnelling, segmentation, and offload occur. Capturing on a client before NAT preserves the client's local tuple; capturing at the server sees the translated or proxied peer. A capture outside TLS sees records but not HTTP fields, while an authorized endpoint capture may see decrypted application data.

On a multi-homed host, `any` can be useful for discovery but may expose a cooked link-layer header rather than Ethernet and can make direction or duplicate observations harder to interpret. A switch SPAN session can omit packets when its destination port is oversubscribed. A physical TAP provides independent wire visibility but still has direction, clock, and capacity limitations.

![Host, switch, and remote captures observe different packet forms and timestamps.](assets/capture-points-offload.svg){fig-alt="Capture points around a host NIC switch mirror and remote host with offload artifacts" width="96%"}

#### **tcpdump and Berkeley Packet Filters**

`tcpdump` uses a libpcap **capture filter** to decide which packets enter the capture path. Filtering early reduces storage and observer overhead. The language combines protocol, host, port, network, direction, and byte-offset predicates. Parentheses should be explicit because a broad filter can silently collect sensitive or irrelevant traffic.

```bash
# Capture only DNS over UDP involving the controlled resolver.
tcpdump -i eth0 -nn -s 0 -w dns-check.pcap \
  'udp port 53 and host 192.0.2.53'

# Read without resolving names; display absolute timestamps and packet details.
tcpdump -nn -tttt -r dns-check.pcap
```

`-nn` avoids reverse name and service lookups that could add traffic or misleading delay. `-s 0` requests the full packet; a smaller **snap length** saves space but may cut off transport or application fields. Rotate bounded files for long captures, restrict file access, and record interface, filter, snap length, clock source, and tool version alongside the PCAP.

Capture filters and Wireshark **display filters** are different languages. `tcp port 443` is a capture filter; `tcp.analysis.retransmission && ip.addr == 192.0.2.10` is a display filter applied after packets are loaded.

#### **Wireshark Dissection and Stream Following**

Wireshark decodes nested protocols using field definitions and conversation state. The packet list summarizes events, the protocol tree exposes fields, and the byte pane ties a decoded field to actual bytes. "Follow stream" reconstructs bytes observed for a conversation; it does not prove that an application consumed them, and retransmissions or missing capture segments can make reconstruction incomplete.

Useful workflow: begin with a narrow time range and endpoints, inspect the handshake, follow sequence and acknowledgment numbers, correlate gaps with retransmissions or duplicate ACKs, then compare with the opposite endpoint. Expert information and generated fields are analysis hints, not ground truth; verify against packet bytes and capture conditions.

![Wireshark presents a packet list, decoded protocol tree, and packet bytes.](assets/wireshark-packets.png){fig-alt="Wireshark packet analysis interface with packet list protocol tree and hexadecimal bytes" width="88%"}

*Figure source: [Drew Carver1, Wireshark packets, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Wireshark_packets.png), licensed under CC BY-SA 4.0.*

#### **Checksums, Offload, Snap Length, and Capture Artifacts**

Modern hosts offload work to the NIC. With checksum offload, a locally captured outgoing packet may contain a placeholder checksum because the NIC fills it later. TCP segmentation offload can make a host capture show a large pseudo-packet that the NIC will split; receive coalescing can merge wire packets before the capture hook. Therefore "bad checksum" or "impossible packet size" near the sender may be an observation artifact. Compare with a wire-side or remote capture and inspect offload settings before concluding corruption.

Other artifacts include dropped capture packets because the capture ring filled, truncated payload because of snap length, duplicate observations across virtual interfaces, changed VLAN tags, clock jumps, and packets reordered by parallel capture processing. Record capture statistics and preserve the raw file.

#### **Scapy for Packet Inspection and Construction**

Scapy represents packets as composable protocol layers and is valuable for controlled labs, test fixtures, and reading PCAP files. A typical offline inspection is `rdpcap("capture.pcap")`, followed by field access such as `packet[IP].src`. Packet construction uses expressions such as `IP(dst="192.0.2.2") / UDP(dport=9999) / Raw(b"probe")`.

Constructing or sending arbitrary packets can bypass normal application safeguards. Use isolated namespaces or emulators, explicit rate limits, and addresses owned by the lab. The executable cell below uses only Python's standard library to build and parse one in-memory Ethernet/IPv4/UDP record, making the wire layout visible without sending traffic or requiring Scapy.

In [3]:
import ipaddress
import struct
from io import BytesIO


def internet_checksum(data):
    """Compute the 16-bit one's-complement checksum used by IPv4."""
    if len(data) % 2:
        data += b"\x00"
    words = struct.unpack(f"!{len(data) // 2}H", data)
    total = sum(words)
    while total >> 16:
        total = (total & 0xFFFF) + (total >> 16)
    return (~total) & 0xFFFF


src_ip = ipaddress.IPv4Address("192.0.2.10").packed
dst_ip = ipaddress.IPv4Address("198.51.100.20").packed
payload = b"measurement-probe"
udp_header = struct.pack("!HHHH", 53000, 9999, 8 + len(payload), 0)

# Build the IPv4 header with a zero checksum, then insert the computed value.
ipv4_without_checksum = struct.pack(
    "!BBHHHBBH4s4s",
    0x45, 0, 20 + len(udp_header) + len(payload), 7, 0,
    64, 17, 0, src_ip, dst_ip,
)
ipv4_header = ipv4_without_checksum[:10] + struct.pack(
    "!H", internet_checksum(ipv4_without_checksum)
) + ipv4_without_checksum[12:]

ethernet_header = bytes.fromhex("00112233445566778899aabb0800")
frame = ethernet_header + ipv4_header + udp_header + payload

# Wrap the frame in a minimal little-endian PCAP structure, entirely in memory.
pcap = BytesIO()
pcap.write(struct.pack("<IHHIIII", 0xA1B2C3D4, 2, 4, 0, 0, 65535, 1))
pcap.write(struct.pack("<IIII", 1_720_000_000, 250_000, len(frame), len(frame)))
pcap.write(frame)

# Parse the same record and expose the fields a packet tool would dissect.
raw = pcap.getvalue()
record_offset = 24
ts_sec, ts_usec, captured_len, original_len = struct.unpack_from("<IIII", raw, record_offset)
packet = raw[record_offset + 16:record_offset + 16 + captured_len]
ethertype = struct.unpack_from("!H", packet, 12)[0]
version_ihl, _, total_len, identification, _, ttl, protocol, checksum, src, dst = struct.unpack_from(
    "!BBHHHBBH4s4s", packet, 14
)
src_port, dst_port, udp_len, _ = struct.unpack_from("!HHHH", packet, 34)

print(f"timestamp: {ts_sec}.{ts_usec:06d}; captured/original: {captured_len}/{original_len}")
print(f"EtherType: 0x{ethertype:04x}; IPv{version_ihl >> 4}; TTL: {ttl}; protocol: {protocol}")
print(f"{ipaddress.IPv4Address(src)}:{src_port} -> {ipaddress.IPv4Address(dst)}:{dst_port}")
print("IPv4 checksum valid:", internet_checksum(packet[14:34]) == 0)
print("payload:", packet[42:].decode())

timestamp: 1720000000.250000; captured/original: 59/59
EtherType: 0x0800; IPv4; TTL: 64; protocol: 17
192.0.2.10:53000 -> 198.51.100.20:9999
IPv4 checksum valid: True
payload: measurement-probe


The parser deliberately separates captured length from original wire length, which is how a PCAP records snap-length truncation. A real protocol analyzer adds link types, options, IPv6 extension headers, fragmentation, transport checksums, malformed-input handling, and stateful reassembly. For production evidence, use maintained parsers rather than extending this teaching parser.

### **Active Measurement Tools**

Active tools are experiments. Each sends a chosen stimulus and observes a response, so interpretation depends on protocol, destination, packet size, rate, time, and vantage point. The most common error is treating a tool's output as a complete map of the network rather than evidence about one controlled exchange.

![ping, traceroute, iperf3, DNS/TLS, and HTTP probes answer different questions and have different blind spots.](assets/active-tools-inference.svg){fig-alt="Five active measurement tools with their question evidence limitations and best use" width="96%"}

#### **ping and ICMP Reachability**

`ping` usually sends ICMP Echo Requests and matches Echo Replies by identifier and sequence number. It reports response ratio and RTT at the sender's clock. A successful reply demonstrates reachability for those ICMP packets at that moment. A timeout is ambiguous: the request, reply, or ICMP processing may have been dropped; a firewall may filter it; the destination may rate-limit control traffic; or the host may be down.

Packet size and the don't-fragment behavior can help investigate MTU problems, but command flags differ by operating system. Always record payload size, interval, count, address family, source interface, and whether DNS names were resolved. Compare the target with intermediate controls such as the local gateway and a known service. An isolated high RTT among otherwise normal samples may reflect ICMP scheduling, while a sustained rise shared by the application deserves corroboration.

```bash
# Bounded, low-rate examples. Flags vary across operating systems.
ping -c 10 -i 1 192.0.2.1
ping -6 -c 10 2001:db8::1
```

#### **traceroute and Path Inference**

Traceroute sends probes with increasing IP TTL or IPv6 Hop Limit. A router that decrements the value to zero may return ICMP Time Exceeded; the destination returns a protocol-dependent terminal response. The reported address is the responder to that probe, and its RTT includes the return path. It is not necessarily the interface that forwarded the probe toward the destination.

Missing stars do not prove packet loss on subsequent forwarding: routers may forward data while filtering or rate-limiting ICMP responses. Per-flow ECMP can send probes with different five-tuples along different paths, creating an apparent topology that no single flow used. NAT, MPLS, tunnels, asymmetric reverse paths, and control-plane scheduling further complicate inference. Repeated probes, a stable flow identifier (the idea behind Paris traceroute), both endpoint captures, and routing telemetry strengthen a path-change conclusion.

![A traceroute output lists hop responses for increasing TTL values.](assets/traceroute-output.png){fig-alt="Terminal output from traceroute showing numbered hops and round trip times" width="78%"}

*Figure source: [Jaho, Traceroute on FreeBSD, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Traceroute.png), released into the public domain.*

In [4]:
from collections import defaultdict


# Synthetic repeated probes. Different responders at the same TTL suggest ECMP
# or response-path variation; a star is not automatically a forwarding failure.
trace_probes = [
    (1, 0, "10.0.0.1", 1.2), (1, 1, "10.0.0.1", 1.1),
    (2, 0, "203.0.113.1", 8.5), (2, 1, "203.0.113.1", 8.7),
    (3, 0, "198.51.100.9", 17.0), (3, 1, "198.51.100.14", 19.4),
    (4, 0, None, None), (4, 1, None, None),
    (5, 0, "192.0.2.80", 31.2), (5, 1, "192.0.2.80", 31.8),
]

by_ttl = defaultdict(list)
for ttl, probe_id, responder, rtt_ms in trace_probes:
    by_ttl[ttl].append((responder, rtt_ms))

for ttl in sorted(by_ttl):
    replies = by_ttl[ttl]
    responders = sorted({address for address, _ in replies if address})
    rtts = [rtt for _, rtt in replies if rtt is not None]
    status = "no ICMP response" if not responders else ", ".join(responders)
    if len(responders) > 1:
        status += "  [multiple responders: test a flow-stable trace]"
    rtt_text = "" if not rtts else f"; median-like RTT {sum(rtts) / len(rtts):.1f} ms"
    print(f"TTL {ttl}: {status}{rtt_text}")

print("\nDestination replied after the silent hop, so TTL 4 did forward the probes.")

TTL 1: 10.0.0.1; median-like RTT 1.1 ms
TTL 2: 203.0.113.1; median-like RTT 8.6 ms
TTL 3: 198.51.100.14, 198.51.100.9  [multiple responders: test a flow-stable trace]; median-like RTT 18.2 ms
TTL 4: no ICMP response
TTL 5: 192.0.2.80; median-like RTT 31.5 ms

Destination replied after the silent hop, so TTL 4 did forward the probes.


#### **iperf and Controlled Throughput Tests**

iperf3 creates a client/server test and measures TCP, UDP, or SCTP transfer behavior. A TCP result reflects the chosen duration, direction, parallel streams, socket buffers, congestion control, host CPU, path, and competing traffic. UDP mode sends at a configured rate and reports receiver-observed loss and jitter; sending far above capacity measures overload behavior, not ordinary capacity.

Useful options include JSON output for reproducibility, reverse direction to test asymmetry, an omitted warm-up interval, and a deliberately chosen number of parallel streams. More streams can fill a high bandwidth-delay path but can also hide a single-flow limitation and compete unfairly with production traffic.

```bash
# Run only between authorized hosts, with a coordinated rate and time window.
iperf3 -s
iperf3 -c 192.0.2.20 -t 20 -O 3 -J
iperf3 -c 192.0.2.20 -R -t 20 -O 3 -J
iperf3 -c 192.0.2.20 -u -b 20M -t 15 -J
```

Schedule high-rate tests, bound duration, monitor link and host guardrails, and stop if user-facing metrics degrade. Keep the raw JSON rather than copying only the final Mbit/s number.

#### **DNS, HTTP, and TLS Timing Measurements**

A complete web probe should preserve phase boundaries. DNS timing includes resolver selection, cache state, transport, recursion, and DNSSEC work. Connection time includes route and transport handshake. TLS time includes cryptographic negotiation and certificate validation. Time to first byte additionally includes request transmission, proxy queueing, and server processing. Body transfer measures object delivery after the first byte.

`curl` can emit selected timings, while `dig` or `kdig` exposes DNS response code, answer, flags, and server. `openssl s_client` can inspect a TLS handshake, but encryption success alone is not a service-identity check unless hostname verification and a trust store are configured. Synthetic probes should use the same hostname, address family, TLS policy, redirects, object size, and region as the user journey.

```bash
curl --silent --output /dev/null \
  --write-out 'dns=%{time_namelookup} connect=%{time_connect} tls=%{time_appconnect} ttfb=%{time_starttransfer} total=%{time_total}\n' \
  https://example.com/health
```

In [5]:
# Synthetic timestamps from one client transaction, in milliseconds from start.
# Converting cumulative timestamps to non-overlapping phases prevents double counting.
events_ms = {
    "start": 0.0,
    "dns_done": 19.0,
    "connect_done": 55.0,
    "tls_done": 92.0,
    "request_sent": 94.0,
    "first_byte": 171.0,
    "body_done": 208.0,
}

phases = {
    "DNS": events_ms["dns_done"] - events_ms["start"],
    "transport connect": events_ms["connect_done"] - events_ms["dns_done"],
    "TLS": events_ms["tls_done"] - events_ms["connect_done"],
    "request + server + first byte": events_ms["first_byte"] - events_ms["request_sent"],
    "response body": events_ms["body_done"] - events_ms["first_byte"],
}

total_explained = sum(phases.values())
unassigned = events_ms["body_done"] - total_explained
for name, duration in phases.items():
    print(f"{name:31s} {duration:6.1f} ms")
print(f"{'client gaps / unassigned':31s} {unassigned:6.1f} ms")
print(f"{'total':31s} {events_ms['body_done']:6.1f} ms")

DNS                               19.0 ms
transport connect                 36.0 ms
TLS                               37.0 ms
request + server + first byte     77.0 ms
response body                     37.0 ms
client gaps / unassigned           2.0 ms
total                            208.0 ms


#### **Safe Rate Limits and Measurement Ethics**

An active measurement has a traffic budget and a social boundary. Obtain authorization, identify operators, specify source addresses and targets, estimate packets and bits per second, limit concurrency, include a stop condition, and avoid collecting content not needed for the question. Prefer documentation ranges in examples and a lab for packet construction. Respect `robots.txt` and service policies where relevant, but remember that neither is a substitute for authorization.

Randomizing targets or distributing probes does not make an unauthorized scan acceptable. Production experiments should begin with the smallest rate capable of answering the question, use exponential backoff on errors, and separate a measurement outage from a service outage. Store the exact command, tool version, start/end time, time zone, and vantage metadata.

### **Passive Measurement and Telemetry**

#### **Interface Counters and System Statistics**

Network interfaces expose cumulative counters such as octets, packets, discards, errors, CRC failures, queue drops, carrier transitions, and sometimes per-priority statistics. A counter value is not a rate. For two readings $C_1$ and $C_2$ separated by $\Delta t$,

$$
r=\frac{C_2-C_1}{\Delta t},
$$

after handling counter wrap, device restart, interface replacement, and missing samples. Compare ingress and egress, packet and byte rates, and neighboring interfaces. An output discard at one device may correspond to no visible error at the receiver because the packet never left the queue.

Host tools add socket state, retransmission counters, queue lengths, CPU, interrupt load, and memory pressure. Correlation is not causation: CPU and network latency can rise together because a service deployment increased both work and traffic.

In [6]:
def counter_delta(previous, current, bits=64, restarted=False):
    """Compute a monotonic counter delta while preserving reset/wrap semantics."""
    if restarted:
        return None, "device restarted; interval is not comparable"
    if current >= previous:
        return current - previous, "normal"
    modulus = 1 << bits
    wrapped_delta = current + modulus - previous
    # A plausible wrap should be close to the counter boundary, not an arbitrary reset.
    if previous > 0.90 * modulus and current < 0.10 * modulus:
        return wrapped_delta, f"{bits}-bit wrap"
    return None, "counter reset or interface replacement"


samples = [
    (4_000_000_000, 4_120_000_000, 64, False),
    ((1 << 32) - 1_000, 4_000, 32, False),
    (9_000_000, 12_000, 64, False),
    (9_000_000, 12_000, 64, True),
]

interval_seconds = 60
for previous, current, width, restarted in samples:
    delta, reason = counter_delta(previous, current, width, restarted)
    rate = "unknown" if delta is None else f"{delta * 8 / interval_seconds / 1e6:.3f} Mbit/s"
    print(f"{previous} -> {current}: {reason}; rate={rate}")

4000000000 -> 4120000000: normal; rate=16.000 Mbit/s
4294966296 -> 4000: 32-bit wrap; rate=0.001 Mbit/s
9000000 -> 12000: counter reset or interface replacement; rate=unknown
9000000 -> 12000: device restarted; interval is not comparable; rate=unknown


#### **NetFlow, IPFIX, and Flow Records**

A flow record summarizes packets sharing a key, commonly source/destination addresses, source/destination ports, protocol, observation point, and time interval. It may include bytes, packets, TCP flags, interfaces, next hop, autonomous system, sampling information, or application labels. NetFlow is a family of implementations; **IPFIX** standardizes templates and information elements so exporters can describe record fields to collectors.

Flows answer "who communicated, when, in which direction, and how much?" far more economically than retaining every packet. They usually cannot reconstruct application messages, exact packet order, or retransmission cause. Active/inactive timeouts split long conversations; NAT and proxies change identities; sampling reduces precision for small flows; templates and exporter resets must be tracked.

![A NetFlow deployment exports summarized flow records from network devices to a collector and analysis system.](assets/netflow-export-architecture.png){fig-alt="NetFlow architecture with exporters collectors and analysis applications" width="78%"}

*Figure source: [Amp 32, NetFlow Architecture 2012, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:NetFlow_Architecture_2012.png), licensed under CC BY-SA 3.0.*

In [7]:
from collections import defaultdict
from ipaddress import ip_network, ip_address


flow_records = [
    {"src": "10.1.0.8", "dst": "198.51.100.20", "dport": 443, "bytes": 82_000, "packets": 90, "sample_n": 1},
    {"src": "10.1.0.8", "dst": "198.51.100.20", "dport": 443, "bytes": 51_000, "packets": 61, "sample_n": 1},
    {"src": "10.1.0.9", "dst": "192.0.2.53", "dport": 53, "bytes": 720, "packets": 8, "sample_n": 1},
    {"src": "10.1.0.10", "dst": "203.0.113.40", "dport": 443, "bytes": 14_000, "packets": 17, "sample_n": 100},
]

internal = ip_network("10.1.0.0/16")
by_external_peer = defaultdict(lambda: {"estimated_bytes": 0, "records": 0})
for record in flow_records:
    # A sampled record is expanded only as an estimate; retain the sampling factor.
    peer = record["dst"] if ip_address(record["src"]) in internal else record["src"]
    bucket = by_external_peer[peer]
    bucket["estimated_bytes"] += record["bytes"] * record["sample_n"]
    bucket["records"] += 1

for peer, totals in sorted(
    by_external_peer.items(), key=lambda item: item[1]["estimated_bytes"], reverse=True
):
    print(peer, totals)

print("\nThe sampled 1-of-100 record is an estimate, not 100 observed copies of the flow.")

203.0.113.40 {'estimated_bytes': 1400000, 'records': 1}
198.51.100.20 {'estimated_bytes': 133000, 'records': 2}
192.0.2.53 {'estimated_bytes': 720, 'records': 1}

The sampled 1-of-100 record is an estimate, not 100 observed copies of the flow.


#### **SNMP and Streaming Telemetry**

SNMP exposes managed objects identified through MIB modules. A manager can poll counters with GET/GETBULK, while agents can send notifications. SNMPv3 adds authentication and privacy modes; earlier community strings should not be treated as strong credentials. Polling is simple and widely supported, but a short event between intervals may disappear into two cumulative readings, and aggressive polling can burden device control planes.

**Streaming telemetry** pushes selected paths when values change or at configured intervals, often using structured models such as YANG. It can provide higher frequency and explicit schemas, but collection must handle backpressure, reconnects, sequence gaps, schema evolution, and high-cardinality labels. "Streaming" does not guarantee lossless or instantaneous evidence.

![SNMP separates managed devices and agents from a manager and management information base.](assets/snmp-architecture.png){fig-alt="SNMP architecture diagram with managed devices agents MIB and manager" width="68%"}

*Figure source: [An.bellizzi / Alb kun, SNMP architecture v.2, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Snmp_architecture_v.2.png), licensed under CC BY-SA 4.0.*

#### **Logs, Traces, and Distributed Correlation**

Logs record discrete events chosen by components. Distributed traces connect work across process boundaries through trace and span identifiers. Metrics summarize repeated observations. Packet and flow evidence show network exchanges. None is universally "more truthful": a server log may prove that code handled a request, a packet capture may prove that bytes crossed a vantage point, and a trace may expose where time was spent across dependencies.

Correlation requires propagated identifiers, synchronized time, stable resource identity, and privacy-aware retention. Preserve both trace ID and the local connection/request context. Proxies may terminate one transport connection and create another, so a five-tuple is not an end-to-end identifier. Never place secrets or unbounded user input in telemetry labels; high-cardinality attributes consume memory and make aggregate queries expensive.

![Network signals pass through collection, normalization, enrichment, retention, and decision stages.](assets/telemetry-evidence-pipeline.svg){fig-alt="Telemetry pipeline from counters flows logs traces and packets to decisions" width="96%"}

In [8]:
from datetime import datetime


events = [
    {"time": "2026-07-20T10:00:00.120+00:00", "source": "client", "trace": "t-104", "message": "request start"},
    {"time": "2026-07-20T10:00:00.151+00:00", "source": "edge", "trace": "t-104", "message": "TLS accepted"},
    {"time": "2026-07-20T10:00:00.169+00:00", "source": "proxy", "trace": "t-104", "message": "origin request start"},
    {"time": "2026-07-20T10:00:00.311+00:00", "source": "origin", "trace": "t-104", "message": "database span complete"},
    {"time": "2026-07-20T10:00:00.338+00:00", "source": "client", "trace": "t-104", "message": "first byte"},
    {"time": "2026-07-20T10:00:00.150+00:00", "source": "edge", "trace": "t-999", "message": "unrelated request"},
]


def parse_timestamp(value):
    return datetime.fromisoformat(value)


trace = sorted(
    (event for event in events if event["trace"] == "t-104"),
    key=lambda event: parse_timestamp(event["time"]),
)
start = parse_timestamp(trace[0]["time"])
for event in trace:
    elapsed_ms = (parse_timestamp(event["time"]) - start).total_seconds() * 1000
    print(f"+{elapsed_ms:6.1f} ms [{event['source']:6s}] {event['message']}")

+   0.0 ms [client] request start
+  31.0 ms [edge  ] TLS accepted
+  49.0 ms [proxy ] origin request start
+ 191.0 ms [origin] database span complete
+ 218.0 ms [client] first byte


### **Measurement Pitfalls**

#### **Clock Synchronization and Timestamp Error**

Suppose a sender timestamps at clock $C_s(t)=a_s t+b_s$ and a receiver at $C_r(t)=a_r t+b_r$. The apparent one-way delay contains true network delay, relative offset $(b_r-b_s)$, and drift from different clock rates. NTP can provide adequate millisecond-level synchronization in many systems, while PTP with hardware timestamping can support tighter requirements on suitable networks. Neither should be assumed: record synchronization state, uncertainty, leap/smear policy, clock steps, and whether timestamps were taken in software, the kernel, the NIC, or dedicated hardware.

RTT avoids cross-host offset by using one clock, but it still includes remote processing and potentially asymmetric paths. For distributed traces, negative or impossible span relationships often reveal clock problems rather than time travel.

In [9]:
from statistics import mean


# Reference exchanges reveal that receiver clock = 1.0002 * true_time + 35 ms.
# Fit a simple line so receiver timestamps can be mapped back to sender time.
reference_pairs = [
    (0.0, 35.0),
    (10_000.0, 10_037.0),
    (20_000.0, 20_039.0),
    (30_000.0, 30_041.0),
]


def linear_fit(points):
    xs = [x for x, _ in points]
    ys = [y for _, y in points]
    x_bar, y_bar = mean(xs), mean(ys)
    slope = sum((x - x_bar) * (y - y_bar) for x, y in points) / sum(
        (x - x_bar) ** 2 for x in xs
    )
    intercept = y_bar - slope * x_bar
    return slope, intercept


clock_rate, clock_offset_ms = linear_fit(reference_pairs)
send_sender_ms = 12_000.0
receive_receiver_ms = 12_064.4
naive_delay = receive_receiver_ms - send_sender_ms
receive_in_sender_time = (receive_receiver_ms - clock_offset_ms) / clock_rate
corrected_delay = receive_in_sender_time - send_sender_ms

print(f"estimated receiver clock: {clock_rate:.7f} * t + {clock_offset_ms:.2f} ms")
print(f"naive one-way delay: {naive_delay:.2f} ms")
print(f"offset/drift-corrected estimate: {corrected_delay:.2f} ms")

estimated receiver clock: 1.0002000 * t + 35.00 ms
naive one-way delay: 64.40 ms
offset/drift-corrected estimate: 26.99 ms


#### **Sampling, Aggregation, and Observer Bias**

Sampling controls cost but changes what can be concluded. Packet sampling favors large flows because they offer more packets to sample. Request sampling may miss rare failures unless errors are retained separately. Head-based trace sampling decides before outcome is known; tail-based sampling can preserve slow/error traces but needs buffering and a complete decision point. Always record the sampling probability and whether it depends on event content.

Aggregation can create Simpson's paradox: every region improves while the global average worsens because traffic shifted toward a naturally slower region. Histograms must use compatible bucket boundaries before aggregation. A dashboard built only from successful server requests excludes connection failures that never reached the server, an example of observer bias.

#### **ICMP Filtering and Rate Limiting**

Routers process normal transit forwarding and locally generated ICMP on different paths. A router can delay or suppress Time Exceeded messages while forwarding application traffic normally. Conversely, successful ping does not prove a particular transport port or MTU works. Use ICMP as one signal, compare protocol-matched probes, inspect endpoint counters, and avoid interpreting a silent intermediate traceroute hop as a failed link.

#### **Load Balancing, Asymmetric Paths, and Middleboxes**

ECMP hashes selected header fields, so two probes can take different next hops. Application load balancers may choose different backends by session, cookie, hostname, or request. Forward and reverse routes are commonly asymmetric, making RTT and one-sided captures hard to localize. NAT, firewalls, proxies, VPNs, and tunnels rewrite or terminate state; correlate pre/post translation identifiers and capture both sides where authorized.

Traceroute is especially vulnerable to false topology inference if each probe changes ports. A path change can also be legitimate while the regression comes from a backend chosen after that path. Treat topology as a hypothesis supported by routing state and repeated flow-stable observations.

#### **Warm-Up, Caching, and Nonstationary Traffic**

The first requests may pay DNS lookup, ARP/ND resolution, TCP/TLS setup, certificate validation, JIT compilation, connection-pool creation, or cache fill. Discarding every first sample hides real cold-start user experience; keeping only cold samples misrepresents steady state. Report them as separate populations.

Network traffic is nonstationary: diurnal load, releases, routing convergence, radio mobility, and incident response change the data-generating process. Randomize experiment order where possible, keep a simultaneous control, record change events, and avoid comparing Monday morning treatment with Sunday night baseline.

In [10]:
# Simpson's paradox: both regions improve, but traffic shifts toward the slower region.
periods = {
    "baseline": {
        "fast-region": {"requests": 900, "mean_ms": 50},
        "slow-region": {"requests": 100, "mean_ms": 200},
    },
    "incident": {
        "fast-region": {"requests": 100, "mean_ms": 45},
        "slow-region": {"requests": 900, "mean_ms": 180},
    },
}


def weighted_mean(groups):
    total_requests = sum(group["requests"] for group in groups.values())
    total_time = sum(group["requests"] * group["mean_ms"] for group in groups.values())
    return total_time / total_requests


for name, groups in periods.items():
    print(name, "global mean:", f"{weighted_mean(groups):.1f} ms")
    for region, values in groups.items():
        print(f"  {region:11s}: {values['mean_ms']} ms across {values['requests']} requests")

print("\nBoth regions became faster, but the global mean rose because the traffic mix changed.")

baseline global mean: 65.0 ms
  fast-region: 50 ms across 900 requests
  slow-region: 200 ms across 100 requests
incident global mean: 166.5 ms
  fast-region: 45 ms across 100 requests
  slow-region: 180 ms across 900 requests

Both regions became faster, but the global mean rose because the traffic mix changed.


### **A Layered Troubleshooting Workflow**

A layered workflow is a search strategy, not a ritual. Begin at the narrowest boundary supported by evidence, compare an affected case with a working control, and stop descending when a layer has been demonstrated healthy for the same population and time. Every step should update the hypothesis ranking.

![Troubleshooting scopes the symptom, tests a falsifiable hypothesis, checks relevant layers, changes one variable, and verifies recovery.](assets/layered-troubleshooting-workflow.svg){fig-alt="Hypothesis-driven layered troubleshooting workflow from symptom scope through link addressing routing transport and service checks" width="96%"}

#### **Define the Symptom and Blast Radius**

Translate "the network is down" into observable behavior: which user action fails, with which error, from which source locations, to which service identity, over which protocol, starting when? The **blast radius** may be one host, subnet, access provider, address family, region, tenant, application version, or all traffic. Ask what still works; a successful control is often more informative than another failure.

Record the incident time in UTC, the first known bad and last known good events, current and recent changes, and an initial user-impact SLI. Avoid prematurely labeling the cause. "Checkout TLS handshakes time out for IPv6 clients in Sydney" is a symptom. "The firewall is broken" is an untested hypothesis.

#### **Check Link and Local Configuration**

At the local boundary, verify administrative and operational interface state, carrier or Wi-Fi association, negotiated speed/duplex, VLAN, MTU, queue drops, CRC errors, and recent physical changes. Compare both ends of a link: errors only on one side can still originate from a cable, optic, receiver, or mismatch. On a host, inspect the actual network namespace and route table used by the application rather than assuming the interactive shell shares its context.

Useful evidence includes `ip -s link`, `ethtool`, switch port counters, wireless signal/retry statistics, and the interface-change log. A rising CRC counter suggests physical corruption; output queue discards with clean CRCs suggest congestion or policy. One static counter value is weaker than a time-aligned delta.

#### **Check Addressing, Neighbor Resolution, and Gateway Reachability**

Confirm address, prefix length, default and policy routes, DHCP lease, duplicate-address detection, and DNS resolver configuration. ARP for IPv4 and Neighbor Discovery for IPv6 resolve on-link next hops; a stale or incomplete neighbor entry can block local delivery even when the default route exists. Verify the selected gateway is on-link under the configured prefix.

A bounded sequence is: check local interface state, inspect the route decision for the exact destination, inspect the neighbor entry for the selected next hop, then test the gateway with a protocol appropriate to the environment. Capturing ARP/ND on the local interface can distinguish "no request sent," "request sent with no answer," and "answer received but state not installed."

```bash
ip address show
ip route get 198.51.100.20
ip neighbor show
```

#### **Check Routing and Path Changes**

The forwarding information base should contain the intended longest-prefix match, next hop, output interface, and policy result for the affected source. In a network device, compare control-plane route, installed forwarding entry, and hardware programming. Check route age, metric, BGP attributes, IGP adjacency, route-policy changes, and whether only IPv4 or IPv6 is affected.

Traceroute can suggest where responses change, but routing tables, flow-stable probes, endpoint captures, and control-plane telemetry are needed to attribute cause. Test MTU and tunnelling hypotheses when small exchanges work but larger ones stall. A route can be valid yet suboptimal; compare RTT and AS/region path with a known-good time or control site.

#### **Check Transport State and Congestion**

For TCP, determine whether failure occurs before SYN, during handshake, after data starts, or during connection teardown. Examine sequence and acknowledgment progression, retransmissions, duplicate ACKs, RTT/RTO, receive window, zero-window events, resets, and connection backlog. A retransmission indicates missing timely acknowledgment at the sender; it does not identify where the packet disappeared.

Separate **congestion** from host limitation. Queue drops, increasing RTT under load, and reduced goodput support congestion. High CPU, socket-buffer pressure, NIC-ring drops, or receiver zero windows support endpoint limitation. Compare one flow with multiple flows only when that distinction is the question. For UDP, application sequence numbers and receiver timestamps are needed because the transport provides no recovery signal.

#### **Check DNS, TLS, and Application Semantics**

DNS failures should be separated into client configuration, transport to resolver, response code, authoritative data, DNSSEC validation, cache state, and address-family selection. TLS failures should distinguish reachability, protocol/cipher negotiation, certificate path, hostname, time validity, and client authentication. HTTP status and application logs then reveal redirects, authorization, overload, dependency failure, or semantic errors.

Testing an origin IP while changing the `Host` header or TLS SNI may route to a different virtual service. A health endpoint can remain green while checkout fails because it bypasses a database. Reproduce the actual method, hostname, address family, TLS policy, authentication context, and payload shape without exposing user secrets.

#### **Change One Variable and Verify Recovery**

An intervention is another experiment. State the expected signal, pre-change baseline, owner, scope, expiry, rollback trigger, and verification window. Change one independent variable where possible: withdraw one route, restore one configuration version, drain one backend, or disable one feature for a canary population. Multiple simultaneous changes may restore service but destroy causal understanding and create hidden state.

Recovery means the original user-facing SLI returns, not merely that a command succeeds. Verify affected and control populations, watch guardrail metrics, confirm no new failure mode appeared, and keep observing through a settling period. If the evidence contradicts the hypothesis, revert the diagnostic change and update the hypothesis list.

In [11]:
from dataclasses import dataclass


@dataclass
class Evidence:
    name: str
    value: object
    supports: tuple
    contradicts: tuple


hypotheses = {
    "local-link": 1.0,
    "routing-path": 1.0,
    "transport-congestion": 1.0,
    "dns": 1.0,
    "tls": 1.0,
    "origin-service": 1.0,
}

observations = [
    Evidence("link carrier and CRC", "up, no CRC increase", (), ("local-link",)),
    Evidence("DNS", "same answer and 18 ms in affected/control", (), ("dns",)),
    Evidence("TCP/TLS", "RTT and handshake unchanged", (), ("routing-path", "transport-congestion", "tls")),
    Evidence("server trace", "database span +310 ms after deploy c42", ("origin-service",), ()),
    Evidence("blast radius", "only checkout requests using new query", ("origin-service",), ("local-link",)),
]

# This is an auditable prioritization aid, not a probabilistic diagnosis.
for evidence in observations:
    for name in evidence.supports:
        hypotheses[name] *= 2.0
    for name in evidence.contradicts:
        hypotheses[name] *= 0.35

print("Ranked hypotheses after current evidence:")
for name, score in sorted(hypotheses.items(), key=lambda item: item[1], reverse=True):
    print(f"  {name:23s} score={score:.3f}")

print("\nNext discriminating test: canary rollback deploy c42 for the affected query path.")
print("Success criterion: checkout p95 and DB span return to baseline without control regression.")

Ranked hypotheses after current evidence:
  origin-service          score=4.000
  routing-path            score=0.350
  transport-congestion    score=0.350
  dns                     score=0.350
  tls                     score=0.350
  local-link              score=0.122

Next discriminating test: canary rollback deploy c42 for the affected query path.
Success criterion: checkout p95 and DB span return to baseline without control regression.


The score is intentionally simple and transparent. Its purpose is to prevent an investigation from ignoring contradictory evidence or repeatedly testing the most familiar layer. A real incident also weighs prior probability, impact, cost, and safety of the next test.

### **Network Management and Configuration**

#### **Inventory, Source of Truth, and Configuration State**

Network automation begins with identity: devices, interfaces, circuits, prefixes, autonomous systems, sites, owners, roles, and dependencies. A **source of truth** stores intended state and ownership, not merely a periodic copy of device text. Distinguish:

- **intended state**: what policy says should exist;
- **deployed configuration**: what was delivered to the device;
- **operational state**: what the device and network are actually doing.

All three can differ. A correct template may not have committed, or a committed route policy may produce no route because a neighbor is down. Inventory must have stable identifiers and lifecycle state; reusing an interface name after hardware replacement can otherwise join unrelated telemetry.

#### **Automation, Templates, and Idempotent Changes**

A deterministic template maps validated data to configuration. Data models such as YANG describe structure and constraints; NETCONF and RESTCONF provide structured operations rather than fragile screen scraping. Structured APIs do not make a change correct, but they allow typed validation, explicit datastores, and machine-readable errors.

An operation is **idempotent** when retrying it converges to the same desired state instead of accumulating side effects. Prefer "ensure this prefix-list contains exactly these entries" over "append this line." Compute a semantic diff, limit the change to owned objects, lock or coordinate concurrent writers, and attach a unique change ID.

#### **Validation, Rollback, and Change Windows**

Validation should progress from cheap to realistic: schema and syntax checks, policy rules, referential integrity, topology/reachability analysis, lab or digital-twin tests, canary deployment, and production guardrails. A change window is not permission to skip evidence; it provides coordination, staffing, and a bounded risk period.

Rollback must be designed before commit. A previous text file may be insufficient if a schema changed, state was created externally, or rollback removes the management path. Transactional capabilities such as candidate configuration, confirmed commit, and automatic timeout reduce exposure to partial or unreachable states. Define what triggers rollback and which user-facing signal confirms restoration.

![A safe change moves from source of truth through diff, validation, canary observation, commit, or rollback.](assets/safe-change-loop.svg){fig-alt="Safe configuration transaction with source of truth diff validation canary observe commit and rollback" width="96%"}

#### **Configuration Verification and Policy Compliance**

Post-deployment verification asks whether intended invariants hold, not only whether the API returned success. Examples include "management interfaces are unreachable from user VLANs," "every external route has an approved origin policy," "both redundant links carry traffic under failure," and "MTU supports the tunnel overhead." Some invariants can be checked from configuration; reachability and behavior require operational evidence.

Continuous compliance should identify owner, severity, exception expiry, and remediation path. Alerting on every textual drift produces fatigue when devices reorder equivalent statements. Normalize configuration or compare semantic models, and distinguish an authorized emergency change from unexplained drift.

In [12]:
from copy import deepcopy


running = {
    "edge-1": {"mtu": 1500, "bgp_max_prefix": 120_000, "telemetry": True},
    "edge-2": {"mtu": 1500, "bgp_max_prefix": 120_000, "telemetry": True},
}
desired = {
    "edge-1": {"mtu": 1492, "bgp_max_prefix": 120_000, "telemetry": True},
    "edge-2": {"mtu": 1492, "bgp_max_prefix": 120_000, "telemetry": True},
}


def semantic_diff(current, target):
    changes = []
    for device, target_state in target.items():
        for field, target_value in target_state.items():
            current_value = current[device].get(field)
            if current_value != target_value:
                changes.append((device, field, current_value, target_value))
    return changes


def validate_target(target):
    problems = []
    for device, state in target.items():
        if not 1280 <= state["mtu"] <= 9216:
            problems.append(f"{device}: invalid MTU")
        if state["bgp_max_prefix"] < 10_000:
            problems.append(f"{device}: implausibly low max-prefix")
        if not state["telemetry"]:
            problems.append(f"{device}: telemetry guardrail disabled")
    return problems


candidate = deepcopy(running)
print("planned diff:")
for change in semantic_diff(candidate, desired):
    print(" ", change)

assert not validate_target(desired)

# Canary edge-1 first. A synthetic guardrail fails, so restore the snapshot.
snapshot = deepcopy(candidate)
candidate["edge-1"] = deepcopy(desired["edge-1"])
canary_guardrails = {"management_reachable": True, "p95_change_ms": 118, "limit_ms": 50}
healthy = (
    canary_guardrails["management_reachable"]
    and canary_guardrails["p95_change_ms"] <= canary_guardrails["limit_ms"]
)

if healthy:
    candidate["edge-2"] = deepcopy(desired["edge-2"])
    decision = "commit expanded change"
else:
    candidate = snapshot
    decision = "rollback canary and investigate MTU/path interaction"

print("decision:", decision)
print("remaining drift after decision:", semantic_diff(candidate, desired))

planned diff:
  ('edge-1', 'mtu', 1500, 1492)
  ('edge-2', 'mtu', 1500, 1492)
decision: rollback canary and investigate MTU/path interaction
remaining drift after decision: [('edge-1', 'mtu', 1500, 1492), ('edge-2', 'mtu', 1500, 1492)]


### **Observability and Reliability Engineering**

#### **Metrics, Logs, Traces, and Packet Evidence**

Observability is the ability to infer relevant internal state from emitted evidence. Metrics are compact and good for trends and alerts. Logs preserve selected event detail. Traces connect causal work across services. Packets expose protocol exchanges at a vantage point. Flow records summarize conversations. Configuration and change history explain intended state. An incident-quality system lets an operator move from a user SLI to exemplars, trace IDs, logs, flows, and a bounded capture without losing time or resource identity.

The signals have different cost. Packet retention is detailed but expensive and privacy-sensitive. Metrics are cheap but can lose rare combinations. Logs can overwhelm storage. Traces require context propagation and sampling. Design telemetry around operational questions, avoid unbounded labels, publish schemas and units, and monitor the monitoring pipeline itself.

#### **SLIs, SLOs, Error Budgets, and Alerts**

A **service-level indicator (SLI)** is a measured proportion or distribution tied to user experience, such as successful DNS resolutions, HTTP requests completed under 300 ms, or packets delivered within a delay bound. A **service-level objective (SLO)** is a target over a window. For a 99.9% success objective, the allowed bad fraction is $1-0.999=0.001$.

An **error budget** converts that fraction into allowable bad events or time. If $N$ eligible events occur,

$$
B_{events}=N(1-SLO).
$$

Burn rate compares current error fraction with the allowed fraction:

$$
\text{burn rate}=\frac{1-\text{observed success fraction}}{1-SLO}.
$$

A burn rate of 10 consumes budget ten times as fast as the objective permits. Multi-window alerts combine a short window that detects fast harm with a longer window that rejects brief noise. Alerts should identify a user-impacting condition and an actionable owner, not merely a low-level threshold.

![A reliability objective creates an error budget whose burn rate can drive multi-window alerts.](assets/slo-error-budget.svg){fig-alt="Service level objective error budget consumption and multi-window burn-rate alert" width="94%"}

In [13]:
def error_budget_report(total, bad, objective):
    allowed_fraction = 1 - objective
    observed_bad_fraction = bad / total
    burn_rate = observed_bad_fraction / allowed_fraction
    allowed_bad = total * allowed_fraction
    remaining = allowed_bad - bad
    return {
        "success_fraction": 1 - observed_bad_fraction,
        "allowed_bad": allowed_bad,
        "remaining_bad_events": remaining,
        "burn_rate": burn_rate,
    }


objective = 0.999
short_window = error_budget_report(total=50_000, bad=210, objective=objective)
long_window = error_budget_report(total=600_000, bad=900, objective=objective)

for label, report in [("5-minute", short_window), ("1-hour", long_window)]:
    print(
        f"{label:8s}: success={report['success_fraction']:.4%}, "
        f"burn={report['burn_rate']:.1f}x, remaining={report['remaining_bad_events']:.0f} events"
    )

# A teaching multi-window condition; production thresholds depend on policy.
alert = short_window["burn_rate"] > 14 and long_window["burn_rate"] > 2
print("multi-window page:", alert)

5-minute: success=99.5800%, burn=4.2x, remaining=-160 events
1-hour  : success=99.8500%, burn=1.5x, remaining=-300 events
multi-window page: False


#### **Fault Domains, Redundancy, and Failure Injection**

Redundancy improves availability only when replicas do not share the same hidden dependency. Map fault domains such as power, rack, switch, fibre path, provider, control plane, DNS zone, certificate authority, region, and operator credential. Two links in the same conduit or two services using the same resolver are not independent for that failure.

Failure injection validates assumptions by introducing a bounded fault and observing user SLIs, failover time, state consistency, and recovery. Begin in a lab, then use the smallest production scope, a canary fault domain, explicit abort thresholds, and on-call coordination. Disable one path rather than creating uncontrolled loss; never run denial-of-service experiments without specific authorization. A successful failover test also verifies return to normal, because failback can expose a second defect.

#### **Incident Timelines and Postmortems**

An incident timeline should distinguish observation time, event time, and report time. Record user impact, evidence, hypotheses, decisions, changes, owners, and verification results. Correlate time zones and note clock uncertainty. The timeline is not a transcript of chat; it explains how system state and operator understanding changed.

A useful postmortem is blameless about individuals but precise about mechanisms and decisions. It identifies contributing conditions, why defenses or detection did not limit impact, what went well, and actions with owners and due dates. Corrective work should prefer system changes such as safer defaults, invariant checks, canaries, capacity, or clearer runbooks over "be more careful."

### **Reproducible Network Experiments**

#### **Mininet and Network Namespaces**

Mininet creates virtual hosts, switches, controllers, and links on one Linux system. Hosts use network namespaces and virtual Ethernet pairs, so they run the real Linux network stack and ordinary programs. This is more realistic than a purely abstract simulator for protocol and configuration experiments, while remaining cheaper and more repeatable than a hardware lab.

The shared host remains a limitation: CPU scheduling, clock behavior, NIC hardware, offload, and physical queueing do not perfectly represent production. Treat Mininet as an emulator of selected mechanisms, not proof that a data-center-scale system will have identical timing.

#### **Controlled Topology, Delay, Loss, and Bandwidth**

Linux traffic control can impose delay, variation, loss, duplication, reordering, rate, and queue limits on selected virtual links. Parameters should model a hypothesis. Independent random packet loss is convenient but does not reproduce burst loss, radio retransmission, or congestion queue dynamics. Direction matters: an impaired reverse path can slow TCP acknowledgments even if forward capacity is high.

![A Mininet experiment combines namespaces, real network stacks, virtual links, controlled impairments, and a recorded manifest.](assets/mininet-reproducible-experiment.svg){fig-alt="Mininet namespaces virtual switch traffic control impairments and experiment manifest" width="96%"}

```python
# Illustrative Mininet structure; run in an authorized Linux lab with Mininet installed.
from mininet.net import Mininet
from mininet.link import TCLink

net = Mininet(link=TCLink)
h1, h2 = net.addHost("h1"), net.addHost("h2")
s1 = net.addSwitch("s1")
net.addLink(h1, s1, delay="20ms", loss=1, bw=10)
net.addLink(s1, h2)
net.start()
# Run bounded tests and save raw output here.
net.stop()
```

#### **Experiment Manifests and Raw Artifacts**

A reproducible manifest records topology, addresses, routes, configuration, impairment direction and parameters, workload, warm-up, duration, random seed, repetitions, versions, hardware/OS context, clock source, and expected outputs. Assign a run ID before execution and preserve stdout/stderr, PCAP, telemetry, logs, and configuration snapshots without manual editing.

Derived tables and charts should be rebuildable from raw artifacts. Store checksums so accidental modification is detectable. Redact secrets through a documented transformation while retaining an untouched protected original when policy permits.

#### **Separating Measurement Error from System Behavior**

Repeat independent runs, randomize treatment order, include a no-impairment control, and measure observer overhead. Confidence intervals quantify sampling uncertainty but do not fix bias. A narrow interval around a clock-biased estimate is precisely wrong. Check whether capture drops, CPU saturation, collector backlog, or synchronized cross traffic changed during the experiment.

Use the experiment's independent unit in analysis. Thousands of packets from one run estimate within-run variation; they do not replace multiple runs when the treatment is applied per topology setup. Report effect size and distribution, not only a p-value.

In [14]:
import hashlib
import json
import math
from statistics import mean, stdev


manifest = {
    "topology": {"hosts": ["h1", "h2"], "switches": ["s1"]},
    "link_treatment": {"direction": "h1->s1", "delay_ms": 20, "loss_percent": 1, "rate_mbps": 10},
    "workload": {"name": "bounded-https-like-transfer", "duration_s": 15, "warmup_s": 3},
    "independent_unit": "fresh topology run",
    "repetitions": 6,
    "random_seed": 5046,
    "environment": {"kernel": "record-at-runtime", "tool_versions": "record-at-runtime"},
}

canonical = json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()
run_id = hashlib.sha256(canonical).hexdigest()[:12]

# One p95 result from each independent topology run, not one row per packet.
treatment_p95_ms = [94.2, 97.1, 92.8, 101.0, 95.5, 98.4]
control_p95_ms = [51.2, 49.8, 52.0, 50.7, 51.5, 50.3]
deltas = [treated - control for treated, control in zip(treatment_p95_ms, control_p95_ms)]

# Small-sample teaching interval using t critical ~= 2.571 for df=5 at 95%.
t_critical = 2.571
standard_error = stdev(deltas) / math.sqrt(len(deltas))
interval = (mean(deltas) - t_critical * standard_error, mean(deltas) + t_critical * standard_error)

print("run ID:", run_id)
print("paired p95 effect:", f"{mean(deltas):.1f} ms")
print("95% interval across independent runs:", f"[{interval[0]:.1f}, {interval[1]:.1f}] ms")

run ID: 6bbd2e82c0ad
paired p95 effect: 45.6 ms
95% interval across independent runs: [41.8, 49.3] ms


### **Diagnosing an End-to-End Performance Regression**

Consider a checkout service whose client-observed p95 rises from 181 ms to 461 ms after deployment `c42`. A disciplined investigation keeps the user population fixed and decomposes the change:

1. **Scope:** only requests using the new checkout query are slow; static content and a control region are healthy.
2. **Client phases:** DNS, transport connection, TLS, and body transfer remain near baseline; time to first byte rises.
3. **Packet evidence:** handshake RTT, retransmission rate, and receive window are unchanged for affected connections.
4. **Trace evidence:** the database span under deployment `c42` grows by roughly 300 ms.
5. **Discriminating action:** a canary backend is rolled back while the rest remains unchanged.
6. **Verification:** canary p95 and database span recover, then a bounded rollback restores the affected population while guardrails remain healthy.

![Baseline and incident waterfalls localize the largest change to server processing rather than the network handshake.](assets/end-to-end-regression-waterfall.svg){fig-alt="Baseline and incident request waterfalls with a large server processing regression" width="96%"}

The network was part of the end-to-end path but not the failing mechanism in this case. That conclusion is stronger than "ping looks fine" because protocol-matched client timing, packet evidence, server traces, deployment history, and a reversible intervention agree.

In [15]:
baseline = {
    "dns": 25, "connect": 40, "tls": 36, "server": 60, "body": 20,
    "retransmit_percent": 0.12, "handshake_rtt_ms": 31,
}
incident = {
    "dns": 26, "connect": 42, "tls": 38, "server": 329, "body": 25,
    "retransmit_percent": 0.13, "handshake_rtt_ms": 32,
}
canary_rollback = {
    "dns": 25, "connect": 41, "tls": 37, "server": 64, "body": 21,
    "retransmit_percent": 0.12, "handshake_rtt_ms": 31,
}


def phase_deltas(reference, observed):
    ignored = {"retransmit_percent", "handshake_rtt_ms"}
    return {
        key: observed[key] - reference[key]
        for key in reference
        if key not in ignored
    }


deltas = phase_deltas(baseline, incident)
largest_phase = max(deltas, key=deltas.get)
network_guardrails_stable = (
    abs(incident["handshake_rtt_ms"] - baseline["handshake_rtt_ms"]) <= 3
    and abs(incident["retransmit_percent"] - baseline["retransmit_percent"]) <= 0.05
)
rollback_server_recovered = abs(canary_rollback["server"] - baseline["server"]) <= 10

print("phase deltas:", deltas)
print("largest regression phase:", largest_phase)
print("network handshake guardrails stable:", network_guardrails_stable)
print("canary rollback restored server phase:", rollback_server_recovered)
print(
    "decision:",
    "expand rollback and investigate deployment c42" if network_guardrails_stable and rollback_server_recovered else "collect more evidence",
)

phase deltas: {'dns': 1, 'connect': 2, 'tls': 2, 'server': 269, 'body': 5}
largest regression phase: server
network handshake guardrails stable: True
canary rollback restored server phase: True
decision: expand rollback and investigate deployment c42


### **Summary**

- Begin with a user-visible question, falsifiable hypothesis, metric with units, baseline, vantage point, independent unit, and predeclared success rule.
- RTT, one-way delay, throughput, goodput, utilization, loss, reordering, jitter, tail latency, and availability describe different properties and require explicit populations and intervals.
- A packet capture is a vantage-point observation. Offload, snap length, capture drops, mirrors, encryption, NAT, and clocks can change what appears in the trace.
- ping, traceroute, iperf3, DNS/TLS probes, and HTTP probes ask different questions; a timeout or response must be interpreted in the protocol and policy context.
- Counters, flows, SNMP/streaming telemetry, logs, traces, and packets trade detail, cost, coverage, and privacy. Record schemas, sampling, gaps, clocks, and resource identity.
- Clock error, sampling, aggregation, ICMP policy, load balancing, middleboxes, caches, warm-up, and changing traffic can produce confident but false explanations.
- Troubleshooting scopes impact, compares affected and healthy controls, ranks hypotheses, checks only relevant layers, changes one variable, and verifies the original SLI.
- Network configuration should be modeled as intended, deployed, and operational state, with semantic diffs, validation, canaries, idempotence, rollback, and post-change invariants.
- SLOs and error budgets connect measurements to user harm and alert urgency; fault-domain testing and postmortems turn individual incidents into system improvement.
- Reproducible experiments preserve manifests, versions, raw artifacts, independent repetitions, observer limitations, and uncertainty.

The next chapter builds on this operational foundation to examine programmable networks, software-defined control, data-center fabrics, overlays, cloud networking, and how automation changes both the scale of control and the possible blast radius.